In [64]:
import re
from konlpy.tag import Okt, Kkma
import pandas as pd

In [65]:
# 정규표현식 연습
raw_text = "안녕하세요!!! 123반갑습니다... @@@자연어 처리는 재미있어요 ^_^"
# 한글과 공백을 제외하고 모두 제거
cleaned_text = re.sub(r'[^가-힣\s]', '',raw_text)
cleaned_text

'안녕하세요 반갑습니다 자연어 처리는 재미있어요 '

In [66]:
# 한국어 형태소 분석기 비교
okt = Okt()
kma = Kkma()
text = '나는 사과를 먹는다.'
print(okt.pos(text))
print(kma.pos(text))

[('나', 'Noun'), ('는', 'Josa'), ('사과', 'Noun'), ('를', 'Josa'), ('먹는다', 'Verb'), ('.', 'Punctuation')]
[('나', 'VV'), ('는', 'ETD'), ('사과', 'NNG'), ('를', 'JKO'), ('먹', 'VV'), ('는', 'EPT'), ('다', 'EFN'), ('.', 'SF')]


In [67]:
# okt 형태소 분석기를이용해서 조사와 구두점을 제거
text = '사과가 맛있다, 나는 사과를 먹는다'
[token for token, pos in okt.pos(text) if pos not in ['Josa','Punctuation']]

['사과', '맛있다', '나', '사과', '먹는다']

In [68]:
# 다음 영화 리뷰데이터셋
# 1. 데이터로드
# 2. 전처리(정규식이용해서 한글과 공백만 추출)
# 3. 형태소 분석 및 품사 필터링(명사 동사 형용사만 추출)

# 4. TTR 계산  (상위 100개 리뷰를 대상으로 전체 토큰수 대비 고유타입의 수를 비율(TTR) 계산)
    #   TTR < 0.5 어휘반복이 많아서 TTR이 낮게 츠정  else  다양한 어휘가 사용되어서 TTR이 높게 책성


In [69]:
# 1. 데이터로드
import pandas as pd
daum_df= pd.read_csv('daum_movie_review.csv')
corpus = daum_df['review'].to_numpy()
corpus

array(['돈 들인건 티가 나지만 보는 내내 하품만',
       '몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.',
       '이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 모은 것까지는 좋았으나 이걸 모두 한 그릇에 섞어버린 듯한 느낌... 그래도 다음 작품을 기대하게 만든다...',
       ..., '가족을 위한 영화... 괜찮은 영화.~~~',
       '간만에 제대로 잘짜여진 각본의 영화를 봤네 여운이 아직도 남아~어른을 위한 애니~',
       '한국개봉을 눈빠지게 기다린 보람이있다 깨우치는게 많은 영화'], dtype=object)

In [70]:
# 2. 전처리(정규식이용해서 한글과 공백만 추출)
import re
cleaned_corpus = [re.sub(r'[^가-힣\s]','',doc)  for doc in corpus]
cleaned_corpus[:5]

['돈 들인건 티가 나지만 보는 내내 하품만',
 '몰입할수밖에 없다 어렵게 생각할 필요없다 내가 전투에 참여한듯 손에 땀이남',
 '이전 작품에 비해 더 화려하고 스케일도 커졌지만 전국 맛집의 음식들을 한데 모은 것까지는 좋았으나 이걸 모두 한 그릇에 섞어버린 듯한 느낌 그래도 다음 작품을 기대하게 만든다',
 '이 정도면 볼만하다고 할 수 있음',
 '재미있다']

In [71]:
# 3. 형태소 분석 및 품사 필터링(명사 동사 형용사만 추출)
def pos_noun_verb_ajective(doc):
    okt = Okt()
    temp = []
    for token , pos in okt.pos(doc):
        if pos in ['Noun','Verb','Adjective'] and len(token)>=2:
            temp.append(token)
    return temp

pos_noun_verb_ajective('.....')

[]

In [72]:
# 3. 형태소 분석 및 품사 필터링(명사 동사 형용사만 추출)
okt = Okt()
cleaned_corpus_pos = []
for copors in cleaned_corpus:
    temp = pos_noun_verb_ajective(copors)
    if len(temp) > 0:
        cleaned_corpus_pos.append(temp)    

In [ ]:
# 4. TTR 계산  (상위 100개 리뷰를 대상으로 전체 토큰수 대비 고유타입의 수를 비율(TTR) 계산)
    #   TTR < 0.5 어휘반복이 많아서 TTR이 낮게 측정  else  다양한 어휘가 사용되어서 TTR이 높게 책성

# 전체토큰

#토큰수 기준 상위 100
import numpy as np
top100_index = np.argsort([-len(doc) for doc in cleaned_corpus_pos])[:100]

top100_corpus = [cleaned_corpus_pos[index] for index in top100_index]

len(set(top100_corpus[0])) / len(top100_corpus[0])

0.8285714285714286